In [0]:
import requests
import re
from urllib.parse import urljoin

def get_current_urls():
    folder = (
        "https://www.nemweb.com.au/"
        "REPORTS/CURRENT/DispatchIS_Reports/"
    )

    html = requests.get(folder).text

    files = re.findall(
        r'PUBLIC_DISPATCHIS_\d{12}_[^"\s<>]*\.zip',
        html
    )

    urls = [
        urljoin(folder, file)
        for file in sorted(set(files))
    ]

    return urls

In [0]:
urls = get_current_urls()

print(f"Found {len(urls)} current files")

for url in urls:
    print(url)

In [0]:
import os
import requests
import time

def download_if_not_exists(url, bronze_folder):
    filename = url.split("/")[-1]
    path = os.path.join(bronze_folder, filename)

    if os.path.exists(path):
        print(f"Skipping: {filename}")
        return path

    print(f"Downloading: {filename}")
    start = time.time()

    response = requests.get(url)
    response.raise_for_status()

    with open(path, "wb") as file:
        file.write(response.content)

    print(f"Finished: {filename} - {time.time() - start:.1f}s")

    return path

In [0]:
bronze_folder = "/Volumes/workspace/default/aemo_mlops_volume/bronze/current"

os.makedirs(bronze_folder, exist_ok=True)

for url in urls:
    download_if_not_exists(url, bronze_folder)

In [0]:
import os
import zipfile
import time

csv_folder = bronze_folder + "_uncompressed"

os.makedirs(csv_folder, exist_ok=True)

zip_files = [
    file for file in os.listdir(bronze_folder)
    if file.endswith(".zip")
]

start = time.time()

print(f"Found {len(zip_files)} ZIP files\n")

for i, filename in enumerate(sorted(zip_files), 1):

    path = os.path.join(bronze_folder, filename)

    with zipfile.ZipFile(path, "r") as zip_file:

        files = zip_file.namelist()

        already_extracted = all(
            os.path.exists(os.path.join(csv_folder, file))
            for file in files
        )

        if already_extracted:
            print(f"[{i}/{len(zip_files)}] Skipping: {filename}")
            continue

        print(f"[{i}/{len(zip_files)}] Extracting: {filename}")

        zip_file.extractall(csv_folder)

print()
print(f"Finished in {time.time() - start:.1f} seconds")